# 06_external_validation_against_tivatrainer

This notebook performs external face validation of the propofol pharmacokinetic reconstruction workflow against TivaTrainer.

Representative induction scenarios are selected from the source dataset and replayed using the same event-driven simulation approach applied in the main analysis. For each scenario, effect-site concentration trajectories are generated under both the Eleveld and Schnider models and compared with corresponding TivaTrainer benchmark outputs.

## Overview

The purpose of this notebook is to confirm that the event parsing, time alignment, unit conversion, and model implementation used in the retrospective reconstruction workflow produce trajectories and peak effect-site concentrations that closely match an established external reference.

## Inputs

This notebook requires a case-level medication dataset containing:
- patient identifiers
- demographic variables
- weight and height
- medication timestamps
- dose or infusion-rate values
- unit labels

## Outputs

- a validation figure comparing reconstructed effect-site concentration trajectories under the Eleveld and Schnider models
- a scenario table comparing peak effect-site concentrations from this workflow with TivaTrainer benchmark values

## Notes

- The scenarios shown here are illustrative validation cases rather than the full analytic cohort.
- Effect-site concentration is model-derived and is not directly measured.
- TivaTrainer comparison values are entered as external benchmark values for the selected scenarios.

## 1. Load source data and define simulation inputs

This section loads the case-level medication dataset used for validation, standardizes time variables, and ensures that height is available in centimetres for model initialization.

In [ ]:
!pip install pyTCI

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PyTCI.models import propofol

# ===========================================================================
# 1. SETUP & DATA LOADING
# ===========================================================================
# Load the case-level dataset (long format)
file_path = "INPUT_DATASET.csv"  # replace with the appropriate local input file
df = pd.read_csv(file_path)

df["ACTION_TIME"] = pd.to_datetime(df["ACTION_TIME"])

# Ensure height is available in cm for model initialization
if "HEIGHT_CM" not in df.columns:
    df["HEIGHT_CM"] = df["HEIGHT_IN"] * 2.54

## 2. Define simulation and formatting helper functions

This section defines the functions used to reconstruct effect-site concentration trajectories for individual cases and to generate compact timeline summaries for the validation table.

The simulation workflow follows the same general event-driven logic used in the main analysis:
- medication events are ordered by timestamp
- times are converted to seconds relative to the first propofol dose
- bolus doses are applied instantaneously
- infusion commands are converted to continuous input rates
- model state is advanced at 1-second resolution

In [ ]:
# ===========================================================================
# 2. Simulation helper functions
# ===========================================================================
def calculate_custom_k21(age):
    """Calculates the Schnider-specific k21 based on age-dependent V2 and CL2."""
    v2 = 18.9 - 0.391 * (age - 53)
    cl2 = (1.29 - 0.024 * (age - 53))
    return (cl2 / v2) / 60 # Convert to per-minute for PyTCI

def run_pk_simulation(df_source, patient_id, model_type='eleveld'):
    """Reconstructs Ce trajectories at 1-second resolution."""
    patient_rows = df_source[df_source['OR_CASE_ID'] == patient_id].sort_values('ACTION_TIME')
    info = patient_rows.iloc[0]

    # Initialize model
    if model_type == 'eleveld':
        model = propofol.Eleveld(age=info['AGE'], weight=info['WEIGHT_KG'],
                                 height=info['HEIGHT_CM'], sex=info['SEX'].lower())
    else:
        model = propofol.Schnider(age=info['AGE'], weight=info['WEIGHT_KG'],
                                  height=info['HEIGHT_CM'], sex=info['SEX'].lower())
        model.k21 = calculate_custom_k21(info['AGE'])

    first_dose_time = patient_rows['ACTION_TIME'].min()
    patient_rows['Rel_Time'] = (patient_rows['ACTION_TIME'] - first_dose_time).dt.total_seconds().astype(int)

    total_seconds = 900
    ce_values = []
    active_inf_rate = 0

    for t in range(total_seconds):
        events = patient_rows[patient_rows['Rel_Time'] == t]
        for _, ev in events.iterrows():
            if 'MG' == str(ev['UNITS']).upper():
                model.give_drug(ev['SIG'])
            elif 'MCG' in str(ev['UNITS']).upper():
                active_inf_rate = (ev['SIG'] * info['WEIGHT_KG']) / 60000

        if active_inf_rate > 0:
            model.give_drug(active_inf_rate)

        model.wait_time(1)
        ce_values.append(model.xeo)

    return pd.DataFrame({'Time': range(total_seconds), 'Ce': ce_values, 'ID': patient_id})

def get_ultra_condensed_timeline(patient_data):
    """Formats the dosing log for the table."""
    first_time = patient_data['ACTION_TIME'].min()
    patient_data = patient_data.sort_values('ACTION_TIME')
    desc = []
    for _, row in patient_data.iterrows():
        rel_t = int((row['ACTION_TIME'] - first_time).total_seconds())
        dose = int(float(row['SIG']))
        if 'MG' == str(row['UNITS']).upper():
            desc.append(f"{dose}mg@{rel_t}s")
        elif 'MCG' in str(row['UNITS']).upper():
            desc.append(f"I:{dose}mcg@{rel_t}s")
    return "; ".join(desc)

## 3. Generate validation trajectories and comparison figure

This section runs the selected validation scenarios under the Eleveld and Schnider models and plots the reconstructed effect-site concentration trajectories.

Solid lines denote Eleveld simulations and dashed lines denote Schnider simulations. The resulting figure is intended to provide a visual comparison of trajectory shape and peak effect-site concentration across the selected benchmark scenarios.

In [ ]:
# ===========================================================================
# 3. FIXED SIMULATION & PLOTTING
# ===========================================================================
# Validation scenarios in display order: [single bolus, bolus plus infusion]
TARGET_IDS = [279084634429353896, 852065194812446020]
TARGET_NAMES = ["Single Bolus", "Bolus + Infusion"]

all_curves = []
colors = ['#1f77b4', '#ff7f0e'] # Blue for Bolus, Orange for Infusion

plt.figure(figsize=(12, 7))

for i, pid in enumerate(TARGET_IDS):
    # Run Both Models
    df_elev = run_pk_simulation(df, pid, model_type='eleveld')
    df_elev['Model'], df_elev['ID'] = 'Eleveld', pid

    df_schn = run_pk_simulation(df, pid, model_type='schnider')
    df_schn['Model'], df_schn['ID'] = 'Schnider', pid

    all_curves.append(df_elev)
    all_curves.append(df_schn)

    # Plotting: Solid = Eleveld, Dashed = Schnider
    plt.plot(df_elev['Time'], df_elev['Ce'], color=colors[i],
             linestyle='-', linewidth=2.5, label=f"Eleveld: {TARGET_NAMES[i]}")
    plt.plot(df_schn['Time'], df_schn['Ce'], color=colors[i],
             linestyle='--', linewidth=2.5, alpha=0.6, label=f"Schnider: {TARGET_NAMES[i]}")

# BJA-Standardized Formatting
plt.title("Validation of reconstructed propofol effect-site trajectories against TivaTrainer")
plt.xlabel("Time (seconds)", fontsize=12)
plt.ylabel(r"Effect-site concentration ($C_{e}$, $\mu\mathrm{g~ml^{-1}}$)", fontsize=12)
plt.legend(loc='upper right', frameon=True, fontsize=10)
plt.grid(True, linestyle=':', alpha=0.6)
plt.xlim(0, 900)

plt.tight_layout()
plt.savefig("figure_validation_comparison.pdf", dpi=300)
plt.show()

## 4. Generate scenario comparison table

This section assembles the validation table for the selected benchmark scenarios.

For each scenario, demographic information, the condensed dosing timeline, and the peak effect-site concentrations generated by this workflow are compared with the corresponding TivaTrainer benchmark values. The final table is exported in a format suitable for manuscript assembly.

In [ ]:
# ===========================================================================
# 4. DATA-DERIVED TABLE GENERATION (TABLE S4)
# ===========================================================================
import pandas as pd

scenario_results = []

# TivaTrainer benchmark peak values for the selected scenarios
# Order: [Line 1 (Bolus), Line 2 (Infusion)]
TIVA_E_PEAKS = [2.4, 3.8]
TIVA_S_PEAKS = [6.1, 8.2]

for i, pid in enumerate(TARGET_IDS):
    # 1. Get demographic data from the source dataframe
    patient_rows = df[df['OR_CASE_ID'] == pid].sort_values('ACTION_TIME')
    info = patient_rows.iloc[0]

    # 2. Derive Sim peaks from the actual simulation data (all_curves)
    # We look for the max 'Ce' for this ID and the specific Model
    e_sim_max = next(c['Ce'].max() for c in all_curves
                     if c['ID'].iloc[0] == pid and c['Model'].iloc[0] == 'Eleveld')

    s_sim_max = next(c['Ce'].max() for c in all_curves
                     if c['ID'].iloc[0] == pid and c['Model'].iloc[0] == 'Schnider')

    # 3. Assemble results with strict sig-fig formatting
    scenario_results.append({
        "Scenario": TARGET_NAMES[i],
        "Age":      f"{int(info['AGE'])}",           # Integer
        "Sex":      info['SEX'].upper()[0],          # F/M
        "Weight":   f"{info['WEIGHT_KG']:.1f}",     # 1 decimal
        "Height":   f"{int(info['HEIGHT_IN'])}",    # Integer
        "Timeline": get_ultra_condensed_timeline(patient_rows),
        "E_Sim":    f"{e_sim_max:.1f}",              # Derived from data
        "E_TIVA":   f"{TIVA_E_PEAKS[i]:.1f}",        # Benchmarked
        "S_Sim":    f"{s_sim_max:.1f}",              # Derived from data
        "S_TIVA":   f"{TIVA_S_PEAKS[i]:.1f}"         # Benchmarked
    })

# Convert to DataFrame
df_table = pd.DataFrame(scenario_results)

# Generate LaTeX fragment (Rows only)
# Header=False and index=False ensures it prints only the content
latex_rows = df_table.to_latex(index=False, header=False, escape=False)

# Remove standard tabular/rule lines so only the raw data rows remain
clean_fragment = ""
for line in latex_rows.split('\n'):
    if any(x in line for x in ['tabular', 'rule']): continue
    if line.strip():
        clean_fragment += line + '\n'

# Save fragment for Overleaf
with open("table_scenarios.tex", "w") as f:
    f.write(clean_fragment.strip())

print("✅ 'table_scenarios.tex' generated using data-derived peak values.")
display(df_table)